# Notebook 05: models and training (Section 9.3, SOP 3)

**Run on the chosen laptop.** Long jobs run as scripts (Section 2.3): `python scripts/run_phase.py --phase 5 --tag` (inside `tmux` in WSL). This notebook calls the same stages, each in its own process so the device is fixed before TensorFlow starts, and shows the results. Every stage is resumable: re-running skips finished configs/seeds.

Dry run anywhere: `PASIG_PROFILE=rehearsal` (tiny settings, outputs under `runs/rehearsal/`, never the real gates).

Stages: compute (G14→G13→G15) → variants (G11, G12) → grid (D12, G19, LSTM grid) → finals (5 seeds) → baselines → thresholds (D11) → checks (G16) → freeze (G17, tag `settings-frozen`).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
# os.environ['PASIG_PROFILE'] = 'rehearsal'   # uncomment for the dry run
import pandas as pd
from pasig import phase5
from pasig.config import out_dir, load
from pasig.gates import require_gates, read_gate
require_gates('G0', 'G6', 'G8', 'G9', 'G10')
TAG = os.environ.get('PASIG_PROFILE', 'full') == 'full'

## 1. Compute gates: G14 determinism → G13 device → G15 seed budget (runs `bench_compute.py` twice per device)

In [ ]:
assert phase5.run(['compute']) == 0
{g: read_gate(g)['decision'] for g in ('G14', 'G13', 'G15')}, load('compute')['plan_hours']

## 2. G11 tidal input variant and G12 output form (reference config, 3 seeds each)

In [ ]:
assert phase5.run(['variants']) == 0
read_gate('G11')['decision'], read_gate('G12')['decision'], load('model_inputs')['g11'], load('model_inputs')['g12']

## 3. Two-stage hybrid grid (24 + 36), G19, standalone LSTM grid (18)

In [ ]:
assert phase5.run(['grid']) == 0
read_gate('G19')['decision'], phase5.state()['hybrid_final'], phase5.state()['lstm_final']

In [ ]:
pd.read_csv(out_dir('runs') / 'grid_stage1.csv').sort_values('val_rmse_mean')[['tag', 'val_rmse_mean', 'epochs', 'seconds']].head(10)

## 4. Final configurations × 5 seeds (determinism on) → frozen checkpoints

In [ ]:
assert phase5.run(['finals']) == 0
pd.read_csv(out_dir('runs') / 'finals.csv')[['kind', 'seed', 'val_rmse_mean', 'epochs', 'best_epoch', 'seconds']]

## 5. Baselines (Table III-4) and classifiers

In [ ]:
assert phase5.run(['baselines']) == 0
pd.read_csv(out_dir('tables') / 'baselines_val_rmse.csv').round(4)

In [ ]:
pd.read_csv(out_dir('tables') / 'classifiers_val.csv').groupby(['model', 'input_set', 'level'])['f1_val'].mean().round(3)

## 6. Trigger offsets (D11) and classifier thresholds → `thresholds.yaml`

In [ ]:
assert phase5.run(['thresholds']) == 0
pd.DataFrame({m: {l: v['offset_m'] for l, v in d.items()} for m, d in load('thresholds')['trigger_offsets'].items()}).T

## 7. G16 persistence sanity (+ bug checklist, Section 9.7)

In [ ]:
assert phase5.run(['checks']) == 0
read_gate('G16')['decision'], pd.read_csv(out_dir('tables') / 'val_rmse_by_model_lead.csv', index_col=0).round(4)

## 8. G17 freeze (`frozen_manifest.json`) and tag `settings-frozen`. Review G16 before running this cell.

In [ ]:
assert phase5.run(['freeze'], tag=TAG) == 0
read_gate('G17')['decision']

## Tests

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_reproducibility.py', 'tests/test_offsets.py', 'tests/test_same_hours.py']) == 0